# *Final Project Part 2: Modeling of Default*
**Chengjia Dong, Junru Wang, Yifan Meng**  
*October 20, 2024*  
*University of Illinois Urbana-Champaign*


## Data Preprocessing

In [28]:
buckets # features we would use to predict default probability

['FICO at Origination',
 'Original UPB',
 'Occupancy Status',
 'Loan Purpose',
 'Debt-To-Income (DTI)',
 'Property State']

### Get Current Loan Delinquency Status Data to See if it default

In [188]:
selected_cols = [1, 39, 43]
chunksize = 10000

# 2015Q1
chunks = pd.read_csv('2015Q1.csv', sep='|', header=None, chunksize=chunksize,
usecols = selected_cols)
delin_2015 = pd.concat(chunks, ignore_index=True)

# 2019Q1
chunks = pd.read_csv('2019Q1.csv', sep='|', header=None, chunksize=chunksize,
usecols = selected_cols)
delin_2019 = pd.concat(chunks, ignore_index=True)

# sort and rename data index
col_names = ['Loan Identifier',           # 2
             'Current Loan Delinquency Status', # 40
             'Zero Balance Code' # 44
]

sorted_pairs = sorted(zip(selected_cols, col_names))
col_sorted, col_names_sorted = zip(*sorted_pairs)
delin_2019.columns = col_names_sorted
delin_2015.columns = col_names_sorted

### Groupby 'Loan Identifier' to get raw data

In [190]:
d_2015 = pd.DataFrame(delin_2015.groupby('Loan Identifier').last())
d_2015['Current Loan Delinquency Status'] = pd.to_numeric(d_2015['Current Loan Delinquency Status'], errors='coerce')
d_2015

,Current Loan Delinquency Status,Zero Balance Code
Loan Identifier,,
100002091588,NaN,1.0
100004219574,0.0,1.0
100004457300,0.0,1.0
100006803390,NaN,1.0
100008763886,NaN,1.0
...,...,...
999989599012,0.0,1.0
999994139983,0.0,NaN
999994368149,0.0,1.0


In [195]:
d_2015 = d_2015[d_2015['Zero Balance Code']!=1.0]
d_2015

,Current Loan Delinquency Status,Zero Balance Code
Loan Identifier,,
100013998431,0.0,NaN
100013999247,0.0,NaN
100014873002,0.0,NaN
100023998128,0.0,NaN
100025408233,0.0,NaN
...,...,...
999957400511,0.0,NaN
999958442691,0.0,NaN
999984720190,0.0,NaN


In [201]:
d_2019 = pd.DataFrame(delin_2019.groupby('Loan Identifier').last())
d_2019['Current Loan Delinquency Status'] = pd.to_numeric(d_2019['Current Loan Delinquency Status'], errors='coerce')
d_2019

,Current Loan Delinquency Status,Zero Balance Code
Loan Identifier,,
100000913397,NaN,1.0
100017539727,NaN,1.0
100018053040,0.0,1.0
100019764317,NaN,1.0
100019765730,NaN,1.0
...,...,...
999980892217,0.0,1.0
999983023544,0.0,1.0
999984004377,NaN,6.0


In [203]:
d_2019 = d_2019[d_2019['Zero Balance Code']!=1.0]
d_2019

,Current Loan Delinquency Status,Zero Balance Code
Loan Identifier,,
100024483476,0.0,NaN
100041001274,0.0,NaN
100044621909,0.0,NaN
100055655278,0.0,NaN
100068035916,0.0,NaN
...,...,...
999956940409,0.0,NaN
999968644223,0.0,NaN
999972096919,0.0,NaN


### Extract Features

In [205]:
labels_2015 = df.groupby('Loan Identifier')[buckets].first()
labels_2015.isna().sum()

FICO at Origination     155
Original UPB              0
Occupancy Status          0
Loan Purpose              0
Debt-To-Income (DTI)     99
Property State            0
dtype: int64

In [97]:
labels_2015

,FICO at Origination,Original UPB,Occupancy Status,Loan Purpose,Debt-To-Income (DTI),Property State
Loan Identifier,,,,,,
100002091588,652.0,345000.0,P,P,32.0,NC
100004219574,732.0,293000.0,P,R,34.0,IL
100004457300,813.0,304000.0,P,P,42.0,GA
100006803390,760.0,110000.0,P,C,28.0,VA
100008763886,734.0,325000.0,P,R,29.0,CA
...,...,...,...,...,...,...
999989599012,722.0,288000.0,P,P,32.0,MT
999994139983,759.0,105000.0,P,C,32.0,AL
999994368149,671.0,111000.0,P,P,40.0,FL


In [235]:
labels_2019 = df2.groupby('Loan Identifier')[buckets].first()
labels_2019

,FICO at Origination,Original UPB,Occupancy Status,Loan Purpose,Debt-To-Income (DTI),Property State
Loan Identifier,,,,,,
100000913397,692.0,324000.0,P,C,49.0,CA
100017539727,722.0,307000.0,P,P,44.0,TX
100018053040,728.0,256000.0,S,P,41.0,NC
100019764317,730.0,248000.0,P,P,40.0,IL
100019765730,727.0,490000.0,P,P,35.0,CA
...,...,...,...,...,...,...
999980892217,814.0,180000.0,I,P,45.0,PA
999983023544,760.0,280000.0,P,C,40.0,WI
999984004377,781.0,155000.0,P,P,36.0,CA


### Merge data and Create a 'default' column to stand for default or not

In [207]:
data2015 = pd.merge(labels_2015, d_2015, left_index=True, right_index=True, how='right') 

def modify_status(status):
    return 1 if status >= 3 else 0

data2015['default'] = data2015['Current Loan Delinquency Status'].apply(modify_status)
data2015

,FICO at Origination,Original UPB,Occupancy Status,Loan Purpose,Debt-To-Income (DTI),Property State,Current Loan Delinquency Status,Zero Balance Code,default
Loan Identifier,,,,,,,,,
100013998431,797.0,129000.0,P,R,34.0,WI,0.0,NaN,0
100013999247,738.0,80000.0,P,C,35.0,IN,0.0,NaN,0
100014873002,741.0,399000.0,P,R,35.0,PA,0.0,NaN,0
100023998128,806.0,125000.0,S,P,49.0,FL,0.0,NaN,0
100025408233,678.0,437000.0,P,C,43.0,CA,0.0,NaN,0
...,...,...,...,...,...,...,...,...,...
999957400511,687.0,104000.0,P,R,32.0,NC,0.0,NaN,0
999958442691,750.0,80000.0,P,C,16.0,OH,0.0,NaN,0
999984720190,790.0,312000.0,P,P,50.0,GA,0.0,NaN,0


In [237]:
data2019 = pd.merge(labels_2019, d_2019, left_index=True, right_index=True, how='right') 

def modify_status(status):
    return 1 if status >= 3 else 0

data2019['default'] = data2019['Current Loan Delinquency Status'].apply(modify_status)
data2019

,FICO at Origination,Original UPB,Occupancy Status,Loan Purpose,Debt-To-Income (DTI),Property State,Current Loan Delinquency Status,Zero Balance Code,default
Loan Identifier,,,,,,,,,
100024483476,732.0,118000.0,P,P,35.0,MS,0.0,NaN,0
100041001274,775.0,225000.0,P,C,40.0,OH,0.0,NaN,0
100044621909,756.0,150000.0,P,C,42.0,NV,0.0,NaN,0
100055655278,777.0,268000.0,I,C,41.0,CA,0.0,NaN,0
100068035916,718.0,484000.0,P,P,31.0,KY,0.0,NaN,0
...,...,...,...,...,...,...,...,...,...
999956940409,755.0,78000.0,P,P,45.0,AR,0.0,NaN,0
999968644223,725.0,223000.0,P,R,50.0,UT,0.0,NaN,0
999972096919,759.0,177000.0,P,P,33.0,TX,0.0,NaN,0


### Process the State label: replace those who's not in ['FL', 'NY', 'TX'] with 'other'

In [213]:
data2015['Property State'] = data2015['Property State'].where(data2015['Property State'].isin(['FL', 'NY', 'TX']), 'other')
data2015['Property State'].unique()

array(['other', 'FL', 'TX', 'NY'], dtype=object)

In [239]:
data2019['Property State'] = data2019['Property State'].where(data2019['Property State'].isin(['FL', 'NY', 'TX']), 'other')
data2019['Property State'].unique()

array(['other', 'NY', 'TX', 'FL'], dtype=object)

## Let's Train model

In [241]:
data2015.drop('Current Loan Delinquency Status',axis = 1,inplace = True)

In [243]:
data2019.drop('Current Loan Delinquency Status',axis = 1,inplace = True)

In [245]:
data2019.dropna(axis = 0,inplace = True)
data2015.dropna(axis = 0,inplace = True)

Our model training includes these steps:  
### 1. Data Preparation
The data is first copied into a new DataFrame, \( \text{data} \), to avoid modifying the original dataset: 

$$
\text{data} = \text{data2015.copy()}
$$

### 2. Feature and Target Definition
We define our features \( X \) and the target variable \( y \). The target variable is labeled as 'default', which indicates whether the loan has defaulted or not.

$$
X = \text{data['FICO at Origination', 'Original UPB', 'Debt-To-Income (DTI)',  
 'Occupancy Status', 'Loan Purpose', 'Property State']}
$$
$$
y = \text{data['default']}
$$

### 3. Train-Test Split
The dataset is then split into training and testing sets using an 80-20 split. This allows us to train the model on one set of data and evaluate its performance on another.

$$
X_{\text{train}}, X_{\text{test}}, y_{\text{train}}, y_{\text{test}} = \text{train\_test\_split}(X, y, \text{test\_size}=0.2, \text{random\_state}=42)
$$

### 4. Feature Transformation
To prepare the data for modeling, we need to transform our features. We separate numerical features from categorical features. Numerical features will be standardized, while categorical features will be one-hot encoded.

$$
\text{numeric\_features} = ['FICO \text{ at } \text{Origination}', 'Original \text{ UPB}', 'Debt-\text{To-} \text{Income (DTI)}']
$$
$$
\text{categorical\_features} = ['Occupancy \text{ Status}', 'Loan \text{ Purpose}', 'Property \text{ State}']
$$

### 5. Column Transformer
We create a ColumnTransformer that applies different preprocessing techniques to the respective feature types.

$$
\text{preprocessor} = \text{ColumnTransformer}(
\text{transformers}=[
    ('num', \text{StandardScaler}(), \text{numeric\_features}),
    ('cat', \text{OneHotEncoder}(), \text{categorical\_features})
])
$$

### 6. Pipeline Creation
A pipeline is constructed that includes the preprocessing steps and the logistic regression classifier. This streamlines the workflow.

$$
\text{pipeline} = \text{Pipeline}( \text{steps}=[
    ('preprocessor', \text{preprocessor}),
    ('classifier', \text{LogisticRegression}())
])
$$

### 7. Model Training
The model is trained using the training data.

$$
\text{pipeline.fit}(X_{\text{train}}, y_{\text{train}})
$$

### 8. Prediction
After training, the model is used to make predictions on the testing set.

$$
y_{\text{pred}} = \text{pipeline.predict}(X_{\text{test}})
$$

### 9. Model Evaluation
Finally, we evaluate the model's performance using accuracy and a classification report, which includes metrics like precision, recall, and F1-score.

$$
\text{Accuracy} = \text{accuracy\_score}(y_{\text{test}}, y_{\text{pred}})
$$
$$
\text{classification\_report}(y_{\text{test}}, y_{\text{pred}})
$$


First, since it's a binary classification problem, let's try Logistic Regression.

In [259]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from imblearn.under_sampling import RandomUnderSampler # undersampling

data = data2015.copy()

# initial X and y
X = data.drop('default', axis=1)
y = data['default']

# undersampler
rus = RandomUnderSampler(random_state=42)
X, y = rus.fit_resample(X, y)

# split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# split features to numeric and categorical ones
numeric_features = ['FICO at Origination', 'Original UPB', 'Debt-To-Income (DTI)']
categorical_features = ['Occupancy Status', 'Loan Purpose', 'Property State']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# Create pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Train
pipeline.fit(X_train, y_train)

# get prediction and assess model
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6071428571428571
              precision    recall  f1-score   support

           0       0.66      0.56      0.60        45
           1       0.57      0.67      0.61        39

    accuracy                           0.61        84
   macro avg       0.61      0.61      0.61        84
weighted avg       0.61      0.61      0.61        84



Decision Tree

In [268]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))  # Using decision Tree
])

# Train
pipeline.fit(X_train, y_train)

# get prediction and assess model
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6547619047619048
              precision    recall  f1-score   support

           0       0.67      0.69      0.68        45
           1       0.63      0.62      0.62        39

    accuracy                           0.65        84
   macro avg       0.65      0.65      0.65        84
weighted avg       0.65      0.65      0.65        84



Random Forest

In [253]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))  # Random Forest
])

# Train
pipeline.fit(X_train, y_train)

# get prediction and assess model
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6071428571428571
              precision    recall  f1-score   support

           0       0.64      0.62      0.63        45
           1       0.57      0.59      0.58        39

    accuracy                           0.61        84
   macro avg       0.61      0.61      0.61        84
weighted avg       0.61      0.61      0.61        84



### Summary

As we can see, in this classification task with six features (three continuous and three categorical), we evaluated three models: logistic regression, random forest, and decision tree. Among them, the decision tree performed best with an accuracy of about 65%, while both logistic regression and random forest achieved around 60% accuracy.

#### Logistic Regression: 
As a linear model, logistic regression struggles with capturing complex, nonlinear relationships between features. Although it's highly interpretable, its limited performance indicates that the dataset likely contains nonlinear patterns that the model cannot capture effectively.

#### Random Forest: 
Despite being an ensemble method designed to improve performance, random forest did not perform as well as expected. This might be due to suboptimal hyperparameter tuning (such as tree depth, number of trees, etc.), resulting in an accuracy similar to logistic regression.

#### Decision Tree: 
The decision tree outperformed the other models with an accuracy of 65%. This suggests that the data contains important nonlinear interactions or patterns that the decision tree is able to capture by adaptively splitting features at different thresholds.

### Future Improvement Directions

#### Hyperparameter Tuning:

Random Forest: Further tuning of hyperparameters, such as the number of trees, maximum depth, and the number of features to consider at each split, could potentially improve the performance of the random forest model.  
Decision Tree: Fine-tuning the tree’s depth or the minimum number of samples required for a split can prevent overfitting or underfitting, which may enhance the model’s generalization ability.

#### Ensemble Learning:

Combining the predictions of multiple models (e.g., using a voting classifier with logistic regression, random forest, and decision tree) could provide a more robust prediction and potentially increase overall accuracy.

In [37]:
# Convert to HTML
from nbconvert import HTMLExporter
import nbformat

with open('Final_Project_Part2.ipynb', encoding='utf-8') as f:
    notebook_content = nbformat.read(f, as_version=4)

html_exporter = HTMLExporter()
(body, resources) = html_exporter.from_notebook_node(notebook_content)

with open('Final_Project_Part2.html', 'w', encoding='utf-8') as f:
    f.write(body)
